# Local Model Inference for MAP Classification: Prompt Benchmarking

Purpose of this Notebook: Explore how well differnt models and different prompts perform on a evaluation set

<div class="alert-warning">
Libraries
</div>

Note: In the following we will use various Large Language Models (LLMs) using the transformers package from HuggingFace. All models, except Phi-mini run on the current latest version (4.57.3). To run the inference or Phi-mini we need to downgrade this package (to version 4.49.0).

In [ ]:
#!pip install transformers==4.49.0 --upgrade #for Phi-mini 

#!pip install --upgrade transformers #for newer models (version 4.57.3 used in this notebook)

Now we can load all the necessary packages.

In [ ]:
# import all necessary libraries
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, pipeline, Gemma3ForConditionalGeneration
from datasets import load_dataset, Dataset
import pandas as pd
import numpy as np
import getpass
from huggingface_hub import login
import json
import re
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
import time
import gc
import os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True' # makes the memory allocator more robust against fragmentation

<div class="alert-warning">
Set the working directory and login to HuggingFace
</div>

Next, we set the working directories on the HPC, define the cache directory for the HuggingFace models, and login to HuggingFace to be able to download the inference models.

In [ ]:
# Set working directory 
os.chdir('../../../../data')

# Define the directory where HuggingFace can save/load the models to
transformers_cache_dir = "../../../../hf_cache"

# Create the directory if it does not exist
if not os.path.exists(transformers_cache_dir):
    os.makedirs(transformers_cache_dir)

### Login to HuggingFace to be able to download local models 
if 'HUGGINGFACE_ACCESS_TOKEN' not in os.environ:
    os.environ['HUGGINGFACE_ACCESS_TOKEN'] = getpass.getpass(prompt='Enter your HuggingFace API key: ')
    
huggingface_access_token = os.environ['HUGGINGFACE_ACCESS_TOKEN']

login(token = huggingface_access_token)

<div class="alert-warning">
Check wether GPU is available for GLLM inference
</div>

Lastly, we will check whether the computing node we are connected to also recognizes the GPU(s) we selected. Depending on the inference model the needed GPU capacity varies. The largest model (Qwen 3 235B Instruct) needs 2xH200 GPUs (about 280 GBs VRAM) for efficient batching.

In [4]:
# Set the random seed to ensure reproducible results across different runs.
torch.random.manual_seed(0)

print("PyTorch CUDA available:", torch.cuda.is_available())
print("CUDA version (from torch):", torch.version.cuda)
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")
print("Cuda device count:", torch.cuda.device_count())

PyTorch CUDA available: True
CUDA version (from torch): 12.1
GPU name: NVIDIA H200
Cuda device count: 2


# Import evaluation set

In this part, we load the evaluation dataset and clean-up the dimension columns. 

In [ ]:
# load evaluation set
df = pd.read_excel('GLLM/evaluation_set_MAP_sentences_final.xlsx') 

# rename adoption to referral
df = df.rename(columns={"Explicit_MAP_adoption": "Explicit_MAP_referral", "Implicit_MAP_adoption": "Implicit_MAP_referral"})

### Harmonization of strings
# Strip whitespace from MAP_dimension columns without affecting NaNs
dimension_cols = [col for col in df.columns if col.startswith("MAP_dimension")]
for col in dimension_cols:
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

# Show the first 3 rows of the dataframe
df.head(3)

# Convert to Hugging Face dataset
dataset = Dataset.from_pandas(df)
dataset

del df

# Testing different Models, i.e. Llama 3, Qwen 3, Gemma 3, and Phi-4

First, we define some helper fuctions, including one that helps to run batched LLM pipeline inference, one to extract the information from the LLM JSON output, and one that evaluates the LLM output.

## (_) Batched Prompt Template


In [ ]:
### Batched function to call the LLM
def batched_prompt_template(batch, system_message, user_message, max_new_tokens, output_column, temperature=0.001):
    sentences = batch['Sentence']

    # Create message batches based on whether a system message is provided    
    if system_message == "":
        messages_batch = [
            [
                {"role": "user", "content": f'{user_message}\n\n Now analyze this <sentence>: {sentence}'}
            ]
            for sentence in sentences
        ]
         
    else:
        messages_batch = [
            [
                {"role": "system", "content": system_message},
                {"role": "user", "content": f'{user_message}\n\n Now analyze this <sentence>: {sentence}'}
            ]
            for sentence in sentences
        ]
    
    # Tokenize the input messages
    input_ids = tokenizer.apply_chat_template(
        messages_batch,
        add_generation_prompt=True,
        return_tensors="pt",
        padding=True,
    ).to(model.device)

    # Create attention mask
    attention_mask = input_ids != tokenizer.pad_token_id

    # Generate outputs
    with torch.no_grad():
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pad_token_id=tokenizer.eos_token_id,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
            )

    # Decode each generated output
    decoded_outputs = tokenizer.batch_decode(outputs[:, input_ids.shape[-1]:], skip_special_tokens=True)

    #free memory
    del input_ids, attention_mask, outputs
    #free gpu memory
    torch.cuda.empty_cache()
    # Return the outputs in the specified format
    return {output_column: [out.strip() for out in decoded_outputs]}

## (_) Extract Json

In [8]:
def extract_json(row):
    """
    Extracts MAP classification fields from LLM output.
    If parsing fails, returns None for all and prints the problematic LLM output.
    """
    llm_output = row["llm_full_output"]
    try:
        json_match = re.search(r'\{.*?\}', llm_output, re.DOTALL)

        if not json_match:
            print("Failed to extract JSON from LLM output:\n", llm_output)
            return {       
                "LLM_Explicit_MAP_referral": None,
                "LLM_Implicit_MAP_referral": None,
                "LLM_Dimension": None,
                "LLM_Confidence_Score": None
            }

        # Parse JSON
        json_str = json_match.group(0)
        result = json.loads(json_str)

        # Ensure the probability is an integer between 0 and 100
        probability = int(result["Confidence_Score"])
        if not (0 <= probability <= 100):
            raise ValueError("Confidence_Score is out of range")

        # Normalize values
        def normalize(value):
            if isinstance(value, str):
                value = value.strip()
                if value.lower() in ["n/a", "", "na", "none", "nan"]:
                    return None
            return value

        return {
            "LLM_Explicit_MAP_referral": normalize(result.get("Explicit_MAP_referral")).capitalize() if normalize(result.get("Explicit_MAP_referral")) else None,
            "LLM_Implicit_MAP_referral": normalize(result.get("Implicit_MAP_referral")).capitalize() if normalize(result.get("Implicit_MAP_referral")) else None,
            "LLM_Dimension": result.get("Dimension") if isinstance(result.get("Dimension"),str) else ", ".join(result.get("Dimension")) if isinstance(result.get("Dimension"), list) else None,
            "LLM_Confidence_Score": probability
        }

    except Exception as e:
        print(f"Error parsing JSON: {e}\nLLM Output:\n{llm_output}\n")
        return {
            "LLM_Explicit_MAP_referral": None,
            "LLM_Implicit_MAP_referral": None,
            "LLM_Dimension": None,
            "LLM_Confidence_Score": None
        }

## (_) Evaluate LLM

In [ ]:
def evaluate_llm(dataset, model_id, system_idx, user_idx,
                          truth_exp_col="Explicit_MAP_referral", pred_exp_col="LLM_Explicit_MAP_referral",
                          truth_imp_col="Implicit_MAP_referral", pred_imp_col="LLM_Implicit_MAP_referral",
                          truth_dimension_col="MAP_dimension_1", pred_dimension_col="LLM_Dimension",
                          prompting_time=None):

    # Convert hf dataset to pd dataframe
    df = dataset.to_pandas()
    
    # Drop rows with missing LLM predictions
    df_exp = df.dropna(subset=[truth_exp_col, pred_exp_col]).copy()
    df_imp = df.dropna(subset=[truth_imp_col, pred_imp_col]).copy()

    print(f"Dropped {len(df)-len(df_exp)} explicit / {len(df)-len(df_imp)} implicit sentences")
    
    #Evaluate Explicit MAP Referral
    print("=== Explicit MAP Referral Evaluation ===")
    if not df_exp.empty:
        exp_accuracy = accuracy_score(df_exp[truth_exp_col], df_exp[pred_exp_col])
        exp_f1_yes = f1_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="Yes", zero_division=0)
        exp_precision_yes = precision_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="Yes", zero_division=0)
        exp_recall_yes = recall_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="Yes", zero_division=0)
        exp_f1_no = f1_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="No", zero_division=0)
        exp_precision_no = precision_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="No", zero_division=0)
        exp_recall_no = recall_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="No", zero_division=0)
        print("\nClassification Report (Explicit):")
        print(classification_report(df_exp[truth_exp_col], df_exp[pred_exp_col], labels=["Yes", "No"], zero_division=0, digits=3))
    else:
        print("No valid rows for Explicit MAP evaluation.")
    
    #Evaluate Implicit MAP Referral
    print("\n=== Implicit MAP Referral Evaluation ===")
    if not df_imp.empty:
        imp_accuracy = accuracy_score(df_imp[truth_imp_col], df_imp[pred_imp_col])
        imp_precision_yes = precision_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="Yes", zero_division=0)
        imp_recall_yes = recall_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="Yes", zero_division=0)
        imp_f1_yes = f1_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="Yes", zero_division=0)
        imp_precision_no = precision_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="No", zero_division=0)
        imp_recall_no = recall_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="No", zero_division=0)
        imp_f1_no = f1_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="No", zero_division=0)
        print("\nClassification Report (Implicit):")
        print(classification_report(df_imp[truth_imp_col], df_imp[pred_imp_col], labels=["Yes", "No"], zero_division=0, digits=3))
    else:
        print("No valid rows for Implicit MAP evaluation.")

    #Evaluate MAP Dimension
    print("\n=== MAP Dimension Evaluation ===")
    filtered_dimension = df[
        (df["LLM_Explicit_MAP_referral"] == "No") &
        (df["LLM_Implicit_MAP_referral"] == "No") &
        (~df["LLM_Dimension"].isna())
    ]
    dim_percentage = len(filtered_dimension)/len(df)*100
    print(f"Number of rows where LLM says 'No' to both Explicit and Implicit MAP referral but MAP Dimension is not None: {dim_percentage:.0f}%")

    # Create a list of all dimension columns
    dimension_cols = [col for col in df.columns if col.startswith("MAP_dimension")]
    
    # Check if the micro / macro F1 score for the dimension column 
    label_space = ["Budgeting / Planning", "Cost", "Financing / Investment", "Operations", "Performance / Internal Reporting", "Risk / Internal Control", "Strategy", "Pricing & Revenue Management"]
    
    def encode_labels(text, label_space):
        labels = [l.strip() for l in text.split(",")]
        return [1 if label in labels else 0 for label in label_space]
    
    # join the true cols without empty cells to one column and encode the true and pred dimension cols

    df["dimension_true"] = df[dimension_cols].apply(lambda row: ", ".join(row.dropna().astype(str)), axis=1)
    df["dimension_true_encoded"] = df["dimension_true"].apply(lambda text: encode_labels(text, label_space))
    df["dimension_pred_encoded"] = df[pred_dimension_col].apply(lambda text: encode_labels(text, label_space) if isinstance(text, str) else [0]*len(label_space))

    dimension_micro_f1 = f1_score(df["dimension_true_encoded"].tolist(), df["dimension_pred_encoded"].tolist(), average="micro", zero_division=0)
    dimension_macro_f1 = f1_score(df["dimension_true_encoded"].tolist(), df["dimension_pred_encoded"].tolist(), average="macro", zero_division=0)
    dimension_accuracy = accuracy_score(df["dimension_true_encoded"].tolist(), df["dimension_pred_encoded"].tolist())
    #full report as table
    dimension_full_report = classification_report(df["dimension_true_encoded"].tolist(), df["dimension_pred_encoded"].tolist(), target_names=label_space, zero_division=0, digits=3)

    # In addition, we check if at least one true dimension is included in the predicted dimensions (which may be a comma-separated string)
    def check_dimension_match(row):
        #true_dim = row[truth_dimension_col]
        true_dims = [row[col] for col in dimension_cols if pd.notnull(row[col])]
        pred_raw = row[pred_dimension_col]
        if all(pd.isnull(true_dims)) and pd.isnull(pred_raw):
            return True  # Both are NaN = match
        elif pd.isnull(pred_raw):
            return False  # No prediction = no match
        else:
            pred_dims = [dim.strip() for dim in pred_raw.split(",")]
            return any(true_dim in pred_dims for true_dim in true_dims)

    if not df.empty:
        print("\nFull per-label Dimension classification report:\n")
        print(dimension_full_report)
        print(f"MAP Dimension Accuracy:{dimension_accuracy:.3f}")
        df["dimension_match"] = df.apply(check_dimension_match, axis=1)
        dimension_accuracy_alternative = df["dimension_match"].mean()
        print(f"MAP Dimension Accuracy (both N/A = match, at least one match): {dimension_accuracy_alternative:.3f}")
    else:
        print("No valid rows for MAP Dimension evaluation.")

    # Return the evaluation results as a dictionary
    return {
        "model_id": model_id,
        "system_idx": system_idx,
        "user_idx": user_idx,
        "dropped_explicit": len(df) - len(df_exp),
        "dropped_implicit": len(df) - len(df_imp),
        "explicit_accuracy": exp_accuracy,
        "explicit_precision_yes": exp_precision_yes,
        "explicit_recall_yes": exp_recall_yes,
        "explicit_f1_yes": exp_f1_yes,
        "explicit_precision_no": exp_precision_no,
        "explicit_recall_no": exp_recall_no,
        "explicit_f1_no": exp_f1_no,
        "implicit_accuracy": imp_accuracy,
        "implicit_precision_yes": imp_precision_yes,
        "implicit_recall_yes": imp_recall_yes,
        "implicit_f1_yes": imp_f1_yes,
        "implicit_precision_no": imp_precision_no,
        "implicit_recall_no": imp_recall_no,
        "implicit_f1_no": imp_f1_no,
        "dimension_percentage_false": dim_percentage,
        "dimension_micro_f1": dimension_micro_f1,
        "dimension_macro_f1": dimension_macro_f1,
        "dimension_accuracy": dimension_accuracy,
        "dimension_accuracy_alternative": dimension_accuracy_alternative,
        "dimension_full_report": dimension_full_report,
        "prompting_time": prompting_time
    }

## Start Zero-Shot Inference

Now, we can define the difference system and user prompts that are used for prompt engineering. In total, we define three system prompts und three user prompts.

In [4]:
### System and User Prompts incl. Implicit, Explicit, Dimension, and Confidence

system_messages = [
  """""",
  """
  You are an AI assistant acting as a Senior Equity Analyst with expertise in Management Accounting Practices (MAP) that analyzes corporate reports for MAP-related methods, tools, and insights.
    
  Please consider the following definition of Management Accounting for your answer:
  "Management accounting is a profession that involves partnering in management decision-making, devising planning and performance management systems,
  and providing expertise in financial reporting and control to assist management in the formulation and implementation of an organization’s strategy."
  """,
  """
  You are an AI assistant acting as a Senior Equity Analyst with expertise in Management Accounting Practices (MAP) that analyzes corporate reports for MAP-related methods, tools, and insights.
    
  Please consider the following definition of Management Accounting for your answer:
  "Management accounting is a profession that involves partnering in management decision-making, devising planning and performance management systems,
  and providing expertise in financial reporting and control to assist management in the formulation and implementation of an organization’s strategy."

  When it comes to different Management Accounting Dimensions consider the following definitions:
  - Budgeting / Planning: Involves preparing and managing budgets, forecasts, and strategic plans to guide resource allocation and align operations with long-term goals. This encompasses business and production planning, budget preparation, and forecasting.
  - Cost: Focuses on measuring, analyzing, and reporting the costs associated with producing goods or services. Activities include cost analysis, application of costing methods, cost allocation, and the development of cost reports and plans to improve efficiency and profitability. 
  - Risk / Internal Control: Covers the identification, assessment, and mitigation of financial and operational risks, as well as the design and assessment of internal control systems. It includes compliance monitoring, risk assessments, and internal audits.
  - Financing / Investment: Relates to decisions about acquiring and allocating financial capital. This includes capital budgeting, investment evaluations, funding strategies, and credit management to support strategic initiatives and long-term growth.
  - Performance / Internal Reporting: Involves measuring and communicating internal performance metrics to support decision-making and continuous improvement. This includes the development of dashboards, KPIs, performance analyses, and internal reports.
  - Strategy: Encompasses activities that support long-term organizational direction, including strategic planning, market positioning, goal setting, and competitive analysis.
  - Operations: Pertains to the planning and monitoring of day-to-day processes that deliver products or services. Management accounting supports operations through inventory management, process improvement, and cost-efficiency initiatives.
  - Pricing & Revenue Management: Involves planning and analyzing pricing strategies and revenue streams. Key areas include product pricing, transfer pricing, revenue forecasting, pricing optimization, and revenue performance analysis.
  """
]
user_messages = [
  """
  You are a Senior Management Accounting Analyst assessing corporate disclosures for Management Accounting Practices (MAP).

  Analyze the <sentence> below and return **only** the following JSON format:
  {
    "Explicit_MAP_referral": "Yes" or "No",
    "Implicit_MAP_referral": "Yes" or "No",
    "Dimension": "One or more values from this list [Budgeting / Planning, Cost, Financing / Investment, Operations, Performance / Internal Reporting, Risk / Internal Control, Strategy, Pricing & Revenue Management] or 'N/A' if both 'Explicit_MAP_referral' and 'Implicit_MAP_referral' are 'No'",
    "Confidence_Score": <an integer between 0 and 100 reflecting confidence that the <sentence> refers to MAPs>
  }
  """,
  """
  You are a Senior Management Accounting Analyst specialized in analyzing corporate disclosure on Management Accounting Practices (MAP).

  Analyze the <sentence> below for relevance to Management Accounting Practices (MAP) using the rules encapsulated in "+++++", then respond **only** in the specified JSON format.
  +++++ [BEGIN OF RULES]

  1. "Explicit_MAP_referral": Output "Yes" if the <sentence> explicitly refers to MAP-related methods, tools or techniques used for internal decision-making, else output "No".

  2. "Implicit_MAP_referral": Output "Yes" if the <sentence> implies the internal use of MAPs for decision-making, even if MAP tools are not explicitly mentioned, else output "No". If "Explicit_MAP_referral" is "Yes", "Implicit_MAP_referral" must also be "Yes".
    
  3. "Dimension": Choose one or more of the following categories (comma-separated in one string) that suits best to the provided <sentence>:
    - "Budgeting / Planning"
    - "Cost"
    - "Financing / Investment"
    - "Operations"
    - "Performance / Internal Reporting"
    - "Risk / Internal Control"
    - "Strategy"
    - "Pricing & Revenue Management"
    **NOTE**: Use "N/A" if both "Explicit_MAP_referral" and "Implicit_MAP_referral" are "No".

  4. "Confidence_Score": Provide your honest confidence score between 0 and 100 representing your confidence that the <sentence> explicitly or implicitly refers to MAPs.

  Return your response **ONLY** in the following JSON format:
  {
    "Explicit_MAP_referral": "Yes" or "No",
    "Implicit_MAP_referral": "Yes" or "No",
    "Dimension": "One or more values from the list of step 3 or 'N/A'",
    "Confidence_Score": <an integer between 0 and 100>
  }
  +++++ [END OF RULES]
  """,
  """
  You are a Senior Management Accounting Analyst specialized in analyzing corporate disclosure on Management Accounting Practices (MAP).

  Analyze the <sentence> below for relevance to Management Accounting Practices (MAP) using the rules encapsulated in "+++++", then respond **only** in the specified JSON format.
  +++++ [BEGIN OF RULES]

  1. "Explicit_MAP_referral": Output "Yes" if the <sentence> explicitly refers to MAP-related methods, tools or techniques used for internal decision-making. 
  Examples include budgeting, cost allocation, forecasting and planning, performance management, internal controls, valuation models (e.g. DCF), hedging programs, or MAP-related systems. 
  Output "No" if the <sentence> focuses solely on external financial reporting, GAAP compliance, legal accruals, or mandatory disclosures without management use.

  2. "Implicit_MAP_referral": Output "Yes" if the <sentence> implies the internal use of MAPs for decision-making, even if MAP tools are not explicitly mentioned. 
  Examples include impairment analysis, sensitivity analysis, performance incentives (e.g. bonus plans), or estimation of (pension) reserves/allowances. 
  Output "No" if the content relates purely to external requirements (e.g., tax, IT security, regulatory laws/regulations/standards) or technical accounting without internal use.
  If "Explicit_MAP_referral" is "Yes", "Implicit_MAP_referral" must also be "Yes".
    
  3. "Dimension": Choose one or more of the following categories (comma-separated in one string) that suits best to the provided <sentence>:
    - "Budgeting / Planning"
    - "Cost"
    - "Financing / Investment"
    - "Operations"
    - "Performance / Internal Reporting"
    - "Risk / Internal Control"
    - "Strategy"
    - "Pricing & Revenue Management"
    **NOTE**: Use "N/A" if both "Explicit_MAP_referral" and "Implicit_MAP_referral" are "No".

  4. "Confidence_Score": Provide your honest confidence score between 0 and 100 representing your confidence that the <sentence> explicitly or implicitly refers to MAPs.
  Use 0 if the sentence is completely unrelated to MAPs, and 100 if MAPs are clearly and explicitly mentioned. Values in between should reflect proportional confidence (e.g., 50 = uncertain or weak implicit relation).

  Return your response **ONLY** in the following JSON format:
  {
    "Explicit_MAP_referral": "Yes" or "No",
    "Implicit_MAP_referral": "Yes" or "No",
    "Dimension": "One or more values from the list of step 3 or 'N/A'",
    "Confidence_Score": <an integer between 0 and 100>
  }
  +++++ [END OF RULES]
  """
]

In [ ]:
# Create folder for zero-shot evaluation results if it does not exist
if not os.path.exists("GLLM/Local_prompting_results"):
    os.makedirs("GLLM/Local_prompting_results")


### select the models and batch sizes to test
# batch size might need to be adjusted based on the model size and available GPU memory. 
# The following are suggested batch sizes used with inference on a single H200 GPU 
# (2x H200 GPUs for "Qwen/Qwen3-235B-A22B-Instruct-2507-FP8"):

######################
# small-sized models #
######################

# batch_size = 512
#model_id = "meta-llama/Llama-3.2-3B-Instruct"
#model_id = "Qwen/Qwen3-4B-Instruct-2507"
#model_id = "google/gemma-3-4b-it"
# batch_size = 256
#model_id = "microsoft/Phi-4-mini-instruct" 

#######################
# medium-sized models #
#######################

# batch_size = 256
#model_id = "meta-llama/Llama-3.1-8B-Instruct"
#model_id = "Qwen/Qwen3-30B-A3B-Instruct-2507"
#model_id = "microsoft/phi-4" 
# batch_size = 128 (100 for prompt combination (2,2))
#model_id = "google/gemma-3-27b-it"

######################
# large-sized models #
######################

# batch_size = 128 (64 for Qwen3-235B-A22B-Instruct if prompt combination (2,2) is used)
#model_id = "meta-llama/Llama-3.3-70B-Instruct"
#model_id = "Qwen/Qwen3-235B-A22B-Instruct-2507-FP8"

model_ids = [
     "meta-llama/Llama-3.2-3B-Instruct", "Qwen/Qwen3-4B-Instruct-2507", "google/gemma-3-4b-it"
]

batch_size = 512 # batch size for different model sizes: 64, 128, 256, 512

# select different prompt combinations to test
prompt_idx = [(1,0), (1,1), (1,2), (2,2), (0,2)] # different combinations of system and user prompts to test
# Explanation of prompt_idx:
# (1,0) = standard system prompt with definition, simple user prompt
# (1,1) = standard system prompt with definition, detailed user prompt with rules
# (1,2) = standard system prompt with definition, most detailed user prompt with rules and examples
# (2,2) = extended system prompt with definition and dimension descriptions, most detailed user prompt with rules and examples
# (0,2) = no system prompt, most detailed user prompt with rules and examples

# to reduce memory usage and speed up performance of the large models, we can use 4-bit quantization
quantization_config = BitsAndBytesConfig(load_in_4bit=True, # maximizing speed and minimizing memory
                                         bnb_4bit_compute_dtype=torch.bfloat16, # computations in bfloat16
                                         bnb_4bit_use_double_quant=True,
                                         bnb_4bit_quant_type= "nf4"
                                         )

#check whether the file output_ZS_bench_final_evaluation.xlsx exists already and load it to continue evaluation, else create empty list
if os.path.exists("GLLM/output_ZS_bench_final_evaluation.xlsx"):
    df_evaluation_results = pd.read_excel("GLLM/output_ZS_bench_final_evaluation.xlsx")
    evaluation_results = df_evaluation_results.to_dict('records')
    print(f"Loaded existing evaluation results with {len(evaluation_results)} entries.") 
else:
    evaluation_results = []

# Apply the prompts to the sentences using different models

for model_id in model_ids:

    tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=transformers_cache_dir)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left" # important for batching since these models are decoder-only architectures

    if model_id == "google/gemma-3-27b-it" or model_id == "google/gemma-3-4b-it":
            
        model = Gemma3ForConditionalGeneration.from_pretrained(
            model_id,
            cache_dir=transformers_cache_dir,
            dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True
        ).eval()

    elif model_id == "meta-llama/Llama-3.3-70B-Instruct":
           
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            cache_dir=transformers_cache_dir,
            dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True,
            _attn_implementation="flash_attention_2",
            quantization_config=quantization_config
        ).eval()

    elif model_id == "microsoft/Phi-4-mini-instruct":
           
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            cache_dir=transformers_cache_dir,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True,
            _attn_implementation="flash_attention_2"
        ).eval()

    else: 
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            cache_dir=transformers_cache_dir,
            dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True,
            _attn_implementation="flash_attention_2"
        ).eval()

    for system_idx, user_idx in prompt_idx:

        system_message = system_messages[system_idx]
        user_message = user_messages[user_idx]

        print(f"Processing model: {model_id}, System prompt idx: {system_idx}, User prompt idx: {user_idx}")

        start_time = time.time()

        output = dataset.map(
            lambda batch: batched_prompt_template(batch, system_message, user_message, max_new_tokens=100, output_column="llm_full_output", temperature=0.001),
            batched=True,
            batch_size=batch_size
        )

        prompting_time = time.time() - start_time
        print(f"Prompting time: {prompting_time / 60:.2f} minutes for {len(dataset)}")

        output = dataset.add_column("llm_full_output", output["llm_full_output"])

        # Step 2: Apply JSON extraction to the LLM's raw output
        output = output.map(extract_json)

        # Step 3: Evaluate the performance

        evaluation_result = evaluate_llm(output, model_id, system_idx, user_idx, prompting_time=prompting_time)

        evaluation_results.append(evaluation_result)


        # Convert to DataFrame and save as Excel
        df = pd.DataFrame(output)

        df.to_excel(f"GLLM/Local_prompting_results/output_ZS_sys{system_idx}_user{user_idx}_{model_id.split('/')[-1]}.xlsx")

    #free GPU memory (unload model to load new one)
    del model, tokenizer
    gc.collect()
    torch.cuda.ipc_collect()
    torch.cuda.empty_cache()


#save final evaluation results
df_evaluation_results = pd.DataFrame(evaluation_results)

df_evaluation_results.to_excel(f"GLLM/evaluation_summary_Local_final.xlsx", index=False)


Loaded existing evaluation results with 40 entries.


Loading checkpoint shards:   0%|          | 0/24 [00:00<?, ?it/s]

Parameter 'function'=<function <lambda> at 0x150060f67d90> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Processing model: Qwen/Qwen3-235B-A22B-Instruct-2507-FP8, System prompt idx: 2, User prompt idx: 2


  0%|          | 0/35 [00:00<?, ?ba/s]